# 100-Box Configuration Smoke Test (Colab)

This notebook runs a minimal test to validate that the main experiment configuration for `SCHEMA_BOXES` with `num_instances=100` is wired correctly.

It checks dataset generation and prompt/token consistency, without running a full model patching experiment.

In [ ]:
# Colab setup: clone repo if needed and install dependencies.
import os

REPO_URL = "https://github.com/yoavgur/mixing-mechanisms.git"
REPO_DIR = "100rep"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd 100rep
!pip -q install transformers torch pandas tqdm numpy

In [ ]:
import sys
import random
import numpy as np
import torch

sys.path.append("CausalAbstraction")

from training import (
    sample_answerable_question_template,
    get_counterfactual_datasets,
    ppkn_simpler_counterfactual_template_split_key_loc,
)
from grammar.task_to_causal_model import multi_order_multi_schema_task_to_lookbacks_generic_causal_model
from grammar.schemas import SCHEMA_BOXES
from transformers import AutoTokenizer
from tasks.dist import format_prompt, to_str_tokens

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Main-experiment configuration (matching repository defaults).
schema = SCHEMA_BOXES
num_instances = 100
num_samples = 5
cat_indices_to_query = [0]
cat_to_query = 1
model_id = "google/gemma-2-2b-it"  # Tokenizer only for this smoke test

causal_model = multi_order_multi_schema_task_to_lookbacks_generic_causal_model(
    [schema], num_instances, num_fillers_per_item=0, fillers=False
)
causal_models = {schema.name: causal_model}

counterfactual_template = ppkn_simpler_counterfactual_template_split_key_loc

train_ds, test_ds, fps = get_counterfactual_datasets(
    None,
    [schema],
    num_samples=num_samples,
    num_instances=num_instances,
    cat_indices_to_query=cat_indices_to_query,
    answer_cat_id=cat_to_query,
    do_assert=True,
    do_filter=False,
    counterfactual_template=counterfactual_template,
    causal_models=causal_models,
    sample_an_answerable_question=sample_answerable_question_template,
)

In [ ]:
# Minimal validation checks for the 100-box setup.
tokenizer = AutoTokenizer.from_pretrained(model_id)
records = train_ds[schema.name][schema.name]

assert len(records) == num_samples, f"Expected {num_samples} samples, got {len(records)}"

for i, record in enumerate(records):
    raw_prompt = record["input"]["raw_input"]
    prompt = format_prompt(tokenizer, raw_prompt)
    prompt_tokens = to_str_tokens(tokenizer, prompt)
    metadata = record["input"]["metadata"]

    answer_indices = [
        idx for idx, tok in enumerate(prompt_tokens)
        if schema.matchers[cat_to_query](tok)
    ]

    assert len(answer_indices) == num_instances, (
        f"Sample {i}: expected {num_instances} answer positions, got {len(answer_indices)}"
    )
    assert metadata["keyload"], f"Sample {i}: empty keyload metadata"
    assert metadata["payload"], f"Sample {i}: empty payload metadata"
    assert 0 <= metadata["dst_index"] < num_instances, (
        f"Sample {i}: dst_index out of range ({metadata['dst_index']})"
    )

print("PASS: 100-box configuration is valid for main experiment dataset generation.")
print(f"Checked {num_samples} samples with num_instances={num_instances}.")

In [ ]:
# Optional integration check: one real patched forward pass.
# This is heavier than the smoke test because it loads a model.
RUN_INTEGRATION = False

if RUN_INTEGRATION:
    from tasks.dist import run_with_cf_hf, _num_layers
    from transformers import AutoModelForCausalLM

    # Use a tiny sample for speed.
    record = records[0]
    prompt = format_prompt(tokenizer, record["input"]["raw_input"])
    cf_prompt = format_prompt(tokenizer, record["counterfactual_inputs"][0]["raw_input"])

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    model.eval()

    layer = min(0, _num_layers(model) - 1)

    with torch.no_grad():
        with run_with_cf_hf(
            model,
            tokenizer,
            prompt,
            cf_prompt,
            layer_idx=layer,
            token_positions=[-1],
            alpha=1.0,
        ):
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
            logits = model(input_ids).logits

    assert logits.ndim == 3 and logits.shape[0] == 1, "Unexpected logits shape"
    print("PASS: Integration check succeeded (patched forward pass ran).")
    print(f"Layer used: {layer}; logits shape: {tuple(logits.shape)}")
else:
    print("Integration check skipped. Set RUN_INTEGRATION = True to run it.")

In [ ]:
# Optional output graph plot (small run).
# This generates a mini patch-effect plot compatible with the main experiment output.
RUN_PLOT = False

if RUN_PLOT:
    import matplotlib.pyplot as plt
    from transformers import AutoModelForCausalLM
    from run_layer_experiments import run_experiment_for_layer
    from plotting import plot_patch_effect

    plot_num_samples = 20  # keep small for Colab speed

    # Reuse loaded model if available; otherwise load it.
    if "model" not in globals() or model is None:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        model.eval()

    df_plot = run_experiment_for_layer(
        model=model,
        tokenizer=tokenizer,
        train_ds=train_ds,
        schema=schema,
        num_instances=num_instances,
        num_samples=min(plot_num_samples, len(records)),
        layer=0,
        cat_to_query=cat_to_query,
        model_id=model_id,
        generate=False,
    )

    fig, ax = plot_patch_effect(df_plot, include_reflexive=True, highest_near_pos=0)
    plt.show()

    print("PASS: Output graph generated.")
    print(df_plot["patch_effect"].value_counts())
else:
    print("Plot cell skipped. Set RUN_PLOT = True to run it.")